# STG 4 — Features semánticas de títulos de proyectos

**Objetivo**: convertir cada `titulo_base` en un vector numérico que capture su significado
y agrupar los títulos en temas interpretables.

**Entrada**: `data/df_modelado.csv`  
**Salida**: `data/df_features_titulo.csv`

**Flujo del notebook**:
1. Cargar títulos únicos
2. Calcular embeddings (vectores semánticos)
3. Explorar agrupamientos con K=10, 15, 20
4. ⚠️ **CHECKPOINT HUMANO**: elegir K
5. Generar resultado final y guardar

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

df = pd.read_csv('../data/df_modelado.csv')
titulos = df['titulo_base'].dropna().unique()
titulos = pd.Series(titulos).reset_index(drop=True)

print(f'Títulos únicos en df_modelado: {len(titulos)}')
titulos.head(10)

## Paso 1 — Embeddings

Cada título se convierte en un vector de 384 números que captura su significado semántico.
El modelo `paraphrase-multilingual-MiniLM-L12-v2` fue entrenado en múltiples idiomas
incluyendo español. La primera vez descarga el modelo (~120 MB); las siguientes lo usa desde caché.

In [ ]:
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print('Calculando embeddings...')
embeddings = modelo.encode(titulos.tolist(), show_progress_bar=True)

print(f'\nShape de la matriz de embeddings: {embeddings.shape}')
print(f'  → {embeddings.shape[0]} títulos × {embeddings.shape[1]} dimensiones')

## Paso 2 — Exploración de agrupamientos (K=10, 15, 20)

Probamos tres valores de K. Para cada uno se muestra el tamaño de cada grupo y los 5 títulos
más representativos (los más cercanos al centro del grupo).

⚠️ **Después de revisar esta celda hay un checkpoint: elegís el K que produce grupos más coherentes.**

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

# Palabras a ignorar al generar etiquetas (stopwords + términos legislativos vacíos)
STOPWORDS = {
    'de', 'la', 'el', 'los', 'las', 'del', 'en', 'y', 'a', 'por', 'para',
    'con', 'se', 'su', 'al', 'un', 'una', 'que', 'o', 'e', 'es', 'no',
    'lo', 'le', 'sus', 'entre', 'sobre', 'como', 'más', 'si', 'ha', 'pero',
    # términos legislativos sin valor semántico
    'ley', 'leyes', 'nacional', 'nacionales', 'artículo', 'articulo',
    'modificación', 'modificacion', 'modif', 'proyecto', 'establece',
    'establecimiento', 'creación', 'creacion', 'regimen', 'régimen',
    'república', 'republica', 'argentina', 'honorable', 'cámara', 'camara',
    'diputados', 'senado', 'nación', 'nacion', 'poder', 'ejecutivo',
}

def etiqueta_grupo(titulos_grupo):
    """Devuelve las 5 palabras más frecuentes del grupo (excluyendo stopwords)."""
    palabras = []
    for t in titulos_grupo:
        tokens = re.findall(r'\b[a-záéíóúüñ]{4,}\b', t.lower())
        palabras.extend([p for p in tokens if p not in STOPWORDS])
    from collections import Counter
    top = Counter(palabras).most_common(5)
    return ' / '.join(w for w, _ in top)


def explorar_k(k, embeddings, titulos):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    centros = km.cluster_centers_

    print(f'\n{"="*60}')
    print(f'  K = {k}')
    print(f'{"="*60}')

    for grupo_id in range(k):
        idx_grupo = np.where(labels == grupo_id)[0]
        titulos_grupo = titulos.iloc[idx_grupo].tolist()

        # Los 5 más cercanos al centro
        distancias = cosine_distances([centros[grupo_id]], embeddings[idx_grupo])[0]
        top5_idx = np.argsort(distancias)[:5]
        representativos = [titulos_grupo[i] for i in top5_idx]

        etiqueta = etiqueta_grupo(titulos_grupo)
        print(f'\n  Grupo {grupo_id:>2} ({len(idx_grupo):>3} títulos) | {etiqueta}')
        for t in representativos:
            print(f'    · {t[:80]}')

    return labels

for k in [10, 15, 20]:
    explorar_k(k, embeddings, titulos)